# <font color="#418FDE" size="6.5" uppercase>**Transfer mit Keras**</font>

>Last update: 20260723.
    
By the end of this Lecture, you will be able to:
- Erklären AlexNet-Ideen wie ReLU, größere Filter, Pooling und Dropout anhand kleiner Modelle. 
- Nutzen MobileNetV2 als CPU-freundlichen eingefrorenen Merkmalsextraktor. 
- Vergleichen Transfer-Learning-Ergebnisse mit kleinen selbst trainierten CNNs. 


## **1. AlexNet Ideen**

### **1.1. AlexNet Architekturideen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_16/Lecture_B/image_01_01.jpg?v=1784812585" width="250">



>* ReLU erleichtert das Training tiefer CNNs
>* Wichtige Muster werden sparsam weitergegeben

>* Große Filter erfassen früh mehr Bildkontext
>* Mehr Kontext kostet zusätzliche Rechenleistung

>* Pooling macht Merkmale verschiebungsrobuster
>* Dropout verhindert Auswendiglernen einzelner Pfade



In [ ]:
#@title Python-Code - AlexNet Architekturideen

# Dieses Beispiel zeigt AlexNet-Ideen mit kleinen Bildern.
# ReLU, Pooling und Dropout verändern Aktivierungen sichtbar.
# Die Grafik vergleicht Signalstärke nach jedem Schritt.

import numpy as np
import matplotlib.pyplot as plt

# Ein synthetisches Bild enthält eine helle Kante.
image = np.zeros((16, 16), dtype=float)
image[:, 8:] = 1.0

# Ein großer Filter betrachtet mehr Kontext gleichzeitig.
large_edge_filter = np.array([
    [-1, -1, 0, 1, 1],
    [-1, -1, 0, 1, 1],
    [-1, -1, 0, 1, 1],
    [-1, -1, 0, 1, 1],
    [-1, -1, 0, 1, 1],
], dtype=float)

# Die Faltung misst lokale Kantenstärke im Bild.
feature_map = np.zeros((12, 12), dtype=float)
for row in range(12):
    for col in range(12):
        patch = image[row:row + 5, col:col + 5]
        feature_map[row, col] = np.sum(patch * large_edge_filter)

# ReLU entfernt negative Antworten und behält starke Treffer.
relu_map = np.maximum(feature_map, 0.0)

# Max-Pooling fasst kleine Bereiche robust zusammen.
pooled_map = np.zeros((6, 6), dtype=float)
for row in range(6):
    for col in range(6):
        patch = relu_map[row * 2:row * 2 + 2, col * 2:col * 2 + 2]
        pooled_map[row, col] = np.max(patch)

# Dropout schaltet beim Training zufällige Aktivierungen aus.
rng = np.random.default_rng(42)
dropout_mask = rng.random(pooled_map.shape) >= 0.35
dropout_map = pooled_map * dropout_mask

# Eine kurze Prüfung macht die Formen nachvollziehbar.
if dropout_map.shape != (6, 6):
    raise ValueError("Die Pooling-Ausgabe sollte 6 mal 6 groß sein.")

# Mittelwerte zeigen, wie jede Architekturidee das Signal verändert.
stage_names = ["Faltung", "ReLU", "Pooling", "Dropout"]
stage_strengths = [
    np.mean(feature_map),
    np.mean(relu_map),
    np.mean(pooled_map),
    np.mean(dropout_map),
]

print("AlexNet-Idee: große Filter finden grobe Kanten.")
print("ReLU setzt negative Antworten auf null.")
print("Pooling verkleinert die Karte von 12x12 auf 6x6.")
print("Dropout deaktiviert hier 35 Prozent der Aktivierungen.")

# Ein Balkendiagramm vergleicht die durchschnittliche Aktivierung.
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(stage_names, stage_strengths, color=["gray", "orange", "green", "red"])
ax.set_title("AlexNet-Architekturideen an einem Mini-Beispiel")
ax.set_xlabel("Verarbeitungsschritt")
ax.set_ylabel("Durchschnittliche Aktivierung")
plt.show()



### **1.2. Tensorformen verstehen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_16/Lecture_B/image_01_02.jpg?v=1784812592" width="250">



>* Bilder werden als Tensoren mit Dimensionen dargestellt
>* Tensorformen zeigen passende Schichten und Merkmale

>* Filter und Pooling verändern Tensorgrößen.
>* ReLU und Dropout verändern Aktivierungswerte.

>* Tensorformen zeigen sinnvolle Architekturentscheidungen
>* Schichten verdichten Bilder zu robusten Merkmalen



In [ ]:
#@title Python-Code - Tensorformen verstehen

# Dieses Beispiel verfolgt Tensorformen in kleinen CNN-Bausteinen.
# Faltung, Pooling, ReLU und Dropout wirken unterschiedlich.
# Die Ausgabe zeigt Formänderungen und unveränderte Formen.

import numpy as np

# Ein Mini-Batch enthält zwei synthetische RGB-Bilder.
batch_size = 2
height = 32
width = 32
channels = 3

images = np.zeros((batch_size, height, width, channels), dtype=np.float32)

# Diese Hilfsfunktion berechnet die räumliche Ausgabegröße.
def conv_output_size(input_size, filter_size, stride, padding):
    return ((input_size + 2 * padding - filter_size) // stride) + 1

# Eine große frühe Faltung betrachtet größere Bildausschnitte.
filter_size = 7
stride = 2
padding = 0

conv_height = conv_output_size(height, filter_size, stride, padding)
conv_width = conv_output_size(width, filter_size, stride, padding)
conv_channels = 8
conv_tensor = np.zeros((batch_size, conv_height, conv_width, conv_channels))

# ReLU verändert Werte, aber nicht die äußere Tensorform.
relu_tensor = np.maximum(conv_tensor, 0)

# Pooling halbiert hier Höhe und Breite der Merkmalskarten.
pool_size = 2
pool_height = conv_height // pool_size
pool_width = conv_width // pool_size
pooled_tensor = np.zeros((batch_size, pool_height, pool_width, conv_channels))

# Dropout maskiert Aktivierungen, behält aber dieselbe Form.
rng = np.random.default_rng(42)
dropout_rate = 0.5
mask = rng.random(pooled_tensor.shape) >= dropout_rate
dropout_tensor = pooled_tensor * mask

# Eine einfache Prüfung macht die Formlogik sichtbar.
if relu_tensor.shape != conv_tensor.shape:
    raise ValueError("ReLU sollte die Tensorform nicht verändern.")

if dropout_tensor.shape != pooled_tensor.shape:
    raise ValueError("Dropout sollte die Tensorform nicht verändern.")

print("Tensorformen in einem kleinen AlexNet-inspirierten Ablauf:")
print(f"Eingabe-Batch: {images.shape}")
print(f"Nach 7x7-Faltung, Stride 2, 8 Filtern: {conv_tensor.shape}")
print(f"Nach ReLU: {relu_tensor.shape}")
print(f"Nach 2x2-Pooling: {pooled_tensor.shape}")
print(f"Nach Dropout: {dropout_tensor.shape}")
print("Merke: Faltung und Pooling ändern Formen; ReLU und Dropout nicht.")



### **1.3. Rechenaufwand verstehen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_16/Lecture_B/image_01_03.jpg?v=1784812590" width="250">



>* Viele Filter und Kanäle erhöhen Rechenaufwand
>* Größere Filter liefern Kontext, kosten Ressourcen

>* Pooling verkleinert Merkmalskarten und spart Rechenzeit
>* Mehr Robustheit, aber weniger feine Details

>* ReLU beschleunigt einfache, tiefe Aktivierungen
>* Dropout stärkt robuste, generalisierende Modelle



In [ ]:
#@title Python-Code - Rechenaufwand verstehen

# Dieses Beispiel schätzt CNN-Rechenaufwand anschaulich.
# Filtergröße und Pooling verändern die Multiplikationen.
# Die Ausgabe vergleicht kleine Architekturentscheidungen.

import numpy as np
import matplotlib.pyplot as plt

# Diese Funktion zählt grob Faltungs-Multiplikationen.
def conv_multiplications(height, width, in_channels, filters, kernel_size):
    return height * width * filters * in_channels * kernel_size * kernel_size

# Wir vergleichen drei einfache CNN-Varianten.
model_names = ["3x3 Filter", "11x11 Filter", "11x11 + Pooling"]

small_filter_cost = conv_multiplications(32, 32, 3, 16, 3)
large_filter_cost = conv_multiplications(32, 32, 3, 16, 11)
pooled_large_cost = conv_multiplications(16, 16, 3, 16, 11)

costs = np.array([small_filter_cost, large_filter_cost, pooled_large_cost])
relative_costs = costs / small_filter_cost

# Eine kurze Prüfung macht die Annahmen sichtbar.
if costs.size != len(model_names):
    raise ValueError("Jede Modellvariante braucht genau einen Kostenwert.")

print("Grobe Multiplikationen pro Faltungsschicht:")
for name, cost, relative in zip(model_names, costs, relative_costs):
    print(f"{name}: {cost:,} Multiplikationen, {relative:.1f}x gegenüber 3x3")

# Das Balkendiagramm zeigt den Unterschied auf einen Blick.
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(model_names, relative_costs, color=["#4C78A8", "#F58518", "#54A24B"])
ax.set_title("Rechenaufwand durch Filtergröße und Pooling")
ax.set_xlabel("CNN-Entscheidung")
ax.set_ylabel("Relativer Aufwand gegenüber 3x3")
ax.set_ylim(0, max(relative_costs) * 1.2)

plt.show()



## **2. Keras Applications**

### **2.1. Einordnung**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_16/Lecture_B/image_02_01.jpg?v=1784812579" width="250">



>* Vortrainierte Modelle erkennen allgemeine Bildmerkmale
>* Transfer Learning spart Daten und Rechenzeit

>* MobileNetV2 läuft effizient auch ohne GPU
>* Eingefrorene Gewichte machen Training schneller

>* Vortrainierte Bausteine erkennen viele relevante Bildmuster
>* MobileNetV2 liefert effiziente, vergleichbare Baselines



### **2.2. MobileNetV2 laden**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_16/Lecture_B/image_02_02.jpg?v=1784812581" width="250">



>* Vortrainierte Gewichte erkennen allgemeine Bildmuster
>* MobileNetV2 liefert kompakte Merkmale für kleine Datensätze

>* Alten Klassifikationskopf weglassen
>* Neuen Klassifikator für eigene Klassen trainieren

>* Vortrainierte Schichten bleiben eingefroren
>* Schnelles Training auch ohne starke Hardware



### **2.3. Eingaben passend skalieren**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_16/Lecture_B/image_02_03.jpg?v=1784812583" width="250">



>* Eingaben müssen MobileNetV2s Erwartungen entsprechen
>* Skalierung erhält vertraute Merkmalsbereiche

>* Bildgröße passend an MobileNetV2 anpassen
>* Pixelwerte korrekt für MobileNetV2 skalieren

>* MobileNetV2 braucht vertraut skalierte Eingaben
>* Gleiche Skalierung ermöglicht faire Modellvergleiche



In [ ]:
#@title Python-Code - Eingaben passend skalieren

# Dieses Beispiel zeigt passende MobileNetV2-Eingabeskalierung.
# Geometrie und Pixelwerte werden getrennt betrachtet.
# Die Ausgabe vergleicht rohe und skalierte Werte.

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Ein kleines synthetisches RGB-Bild ersetzt externe Bilddateien.
height = 80
width = 120
x_values = np.linspace(0, 255, width, dtype=np.uint8)

red_channel = np.tile(x_values, (height, 1))
green_channel = np.flipud(red_channel)
blue_channel = np.full((height, width), 90, dtype=np.uint8)
raw_image = np.stack([red_channel, green_channel, blue_channel], axis=2)

# MobileNetV2 erwartet typischerweise quadratische RGB-Eingaben.
target_size = (96, 96)
pil_image = Image.fromarray(raw_image, mode="RGB")
resized_image = np.array(pil_image.resize(target_size, Image.Resampling.BILINEAR))

# Diese Funktion bildet Pixelwerte von 0 bis 255 auf etwa -1 bis 1 ab.
scaled_image = resized_image.astype(np.float32) / 127.5 - 1.0

# Eine einfache Prüfung macht die erwartete Eingabeform sichtbar.
expected_shape = (96, 96, 3)
if resized_image.shape != expected_shape:
    raise ValueError("Die Bildform passt nicht zur erwarteten RGB-Eingabe.")

print("Rohbild-Form:", raw_image.shape)
print("Angepasste Form:", resized_image.shape)
print("Rohwerte min/max:", int(resized_image.min()), int(resized_image.max()))
print("Skalierte Werte min/max:", round(float(scaled_image.min()), 2), round(float(scaled_image.max()), 2))
print("Beispielpixel roh:", resized_image[48, 48].tolist())
print("Beispielpixel skaliert:", np.round(scaled_image[48, 48], 2).tolist())

# Das Bild bleibt visuell gleich, aber die Zahlen ändern sich.
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(resized_image)
ax.set_title("Synthetisches RGB-Bild nach Größenanpassung")
ax.set_xlabel("Breite in Pixeln")
ax.set_ylabel("Höhe in Pixeln")
plt.show()



## **3. Transfer Learning Vergleich**

### **3.1. Basis einfrieren**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_16/Lecture_B/image_03_01.jpg?v=1784812587" width="250">



>* Eingefrorene Basis nutzt gelernte Bildmerkmale
>* Weniger Training, schneller und stabiler

>* Kleines CNN lernt ohne Vorwissen
>* Transfer-Basis nutzt robuste gelernte Merkmale

>* Eingefrorene Basis macht Vergleiche fairer
>* Gute Ausgangslinie, aber nicht immer überlegen



### **3.2. Klassifikationskopf trainieren**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_16/Lecture_B/image_03_02.jpg?v=1784812594" width="250">



>* Vortrainierte Basis liefert robuste Bildmerkmale
>* Kopf lernt neue Klassen effizient

>* Transfer nutzt robuste, vortrainierte Bildmerkmale
>* Kleine CNNs lernen alles selbst

>* Validierungsdaten zeigen Überanpassung besser.
>* Transfer hilft oft, aber nicht immer.



In [ ]:
#@title Python-Code - Klassifikationskopf trainieren

# Dieses Beispiel simuliert einen eingefrorenen Merkmalsextraktor.
# Nur der Klassifikationskopf lernt die Zielklassen.
# Die Testgenauigkeit zeigt den Transfer-Vorteil.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
import sklearn

# Wir nutzen kleine Ziffernbilder als übersichtliche Bildaufgabe.
digits = load_digits()
images = digits.images
target = digits.target

# Diese Prüfung macht die erwartete Bildform explizit.
if images.shape[1:] != (8, 8):
    raise ValueError("Die Ziffernbilder sollten 8 mal 8 Pixel haben.")

# Der Split trennt Lernen und ehrliche Bewertung.
train_images, test_images, y_train, y_test = train_test_split(
    images, target, test_size=0.3, stratify=target, random_state=42
)

# Rohpixel stehen für ein kleines Modell ohne vortrainierte Merkmale.
raw_train = train_images.reshape(len(train_images), -1)
raw_test = test_images.reshape(len(test_images), -1)

# Einfache Bildstatistiken simulieren wiederverwendete robuste Merkmale.
train_top = train_images[:, :4, :].mean(axis=(1, 2))
train_bottom = train_images[:, 4:, :].mean(axis=(1, 2))
train_left = train_images[:, :, :4].mean(axis=(1, 2))
train_right = train_images[:, :, 4:].mean(axis=(1, 2))

# Der Klassifikationskopf sieht nur diese kompakten Merkmale.
transfer_train = np.column_stack(
    [train_top, train_bottom, train_left, train_right]
)

# Dieselben Merkmale werden unverändert für Testbilder berechnet.
test_top = test_images[:, :4, :].mean(axis=(1, 2))
test_bottom = test_images[:, 4:, :].mean(axis=(1, 2))
test_left = test_images[:, :, :4].mean(axis=(1, 2))
test_right = test_images[:, :, 4:].mean(axis=(1, 2))

# Auch der Testkopf erhält nur vier Merkmale pro Bild.
transfer_test = np.column_stack(
    [test_top, test_bottom, test_left, test_right]
)

# Skalierung wird nur auf Trainingsdaten gelernt.
raw_scaler = StandardScaler()
raw_train_scaled = raw_scaler.fit_transform(raw_train)
raw_test_scaled = raw_scaler.transform(raw_test)

# Die Transfer-Merkmale werden ebenfalls sauber skaliert.
transfer_scaler = StandardScaler()
transfer_train_scaled = transfer_scaler.fit_transform(transfer_train)
transfer_test_scaled = transfer_scaler.transform(transfer_test)

# Dieses kleine Modell lernt direkt aus allen Pixeln.
small_cnn_proxy = KNeighborsClassifier(n_neighbors=3)
small_cnn_proxy.fit(raw_train_scaled, y_train)

# Dieser Kopf lernt nur auf eingefrorenen Merkmalen.
classification_head = LogisticRegression(max_iter=300, random_state=42)
classification_head.fit(transfer_train_scaled, y_train)

# Die Testdaten zeigen den Vergleich außerhalb des Trainings.
raw_accuracy = accuracy_score(y_test, small_cnn_proxy.predict(raw_test_scaled))
transfer_accuracy = accuracy_score(
    y_test, classification_head.predict(transfer_test_scaled)
)

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Rohpixel-Modell Testgenauigkeit: {raw_accuracy:.3f}")
print(f"Klassifikationskopf Testgenauigkeit: {transfer_accuracy:.3f}")

# Das Balkendiagramm macht den Vergleich sofort sichtbar.
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(["Rohpixel-Modell", "Klassifikationskopf"], [raw_accuracy, transfer_accuracy])
ax.set_ylim(0, 1)
ax.set_ylabel("Testgenauigkeit")
ax.set_title("Vergleich: direktes Lernen gegen kompakten Kopf")
plt.show()



### **3.3. Transfer Projekt**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_16/Lecture_B/image_03_03.jpg?v=1784812596" width="250">



>* Kleines CNN lernt Bildmuster ohne Vorwissen
>* Transfermodell nutzt Vorwissen und lernt schneller

>* Faire Datenbasis für beide Modelle wählen
>* Lernkurven, Testleistung und Fehler vergleichen

>* Ergebnisse kritisch statt siegerorientiert deuten
>* Modellwahl nach Genauigkeit, Aufwand und Robustheit



# <font color="#418FDE" size="6.5" uppercase>**Transfer mit Keras**</font>


In this lecture, you learned to:
- Erklären AlexNet-Ideen wie ReLU, größere Filter, Pooling und Dropout anhand kleiner Modelle. 
- Nutzen MobileNetV2 als CPU-freundlichen eingefrorenen Merkmalsextraktor. 
- Vergleichen Transfer-Learning-Ergebnisse mit kleinen selbst trainierten CNNs. 

In the next Module (Module 17), we will go over 'PyTorch Grundlagen'